# 26 - Agentic Anomaly Detection over Structured Data Streams
**Stack:** LangGraph ReAct · DuckDB · Ollama · Python 3.12
**Adapted from:** Multi-source sensor track analysis (operational AI context)

A LangGraph ReAct agent that autonomously triages structured data streams
to surface anomalies for an analyst. Demonstrates:
- **Tool-calling agent** over a DuckDB data store
- **Five typed tools** for different analysis operations
- **Ground truth validation** with precision/recall/F1
- **Audit trail** via inspectable message history
- **Local LLM** (Ollama) -- no API key required

**Agent OS relevance:** This is exactly the pattern Agent OS needs --
a domain-specific agent using typed tools over a structured data source,
with observable reasoning and measurable output quality.

In [ ]:
# Install: pip install langgraph langchain-core langchain-ollama duckdb

import duckdb, json, time, random, math
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

OLLAMA_MODEL = 'gpt-oss:20b'
OLLAMA_BASE  = 'http://localhost:11434'

print('Setup OK')

In [ ]:
# Generate synthetic data: 32 tracks x 100 timesteps, 5 behavioral classes
random.seed(42)
DB = duckdb.connect(':memory:')

TRACK_CLASSES = {
    'noise':       (0,  7,  'steady isolated tracks'),
    'maneuver':    (7,  14, 'single track with heading jinks'),
    'formation':   (14, 21, 'tracks flying in close proximity'),
    'geofence':    (21, 26, 'tracks crossing restricted zone'),
    'coordinated': (26, 32, 'simultaneous multi-track heading changes'),
}

rows = []
for track_id in range(32):
    cls = next(c for c, (lo, hi, _) in TRACK_CLASSES.items() if lo <= track_id < hi)
    lat0, lon0 = 35.0 + random.uniform(-0.5, 0.5), -97.0 + random.uniform(-0.5, 0.5)
    hdg = random.uniform(0, 360)
    spd = random.uniform(200, 500)
    for t in range(100):
        ts = 36000 + t * 10
        if cls == 'maneuver' and 30 <= t <= 50:
            hdg += random.uniform(-30, 30)
        if cls == 'geofence' and t == 40:
            lat0, lon0 = 35.5, -96.5  # enter restricted zone
        if cls == 'coordinated' and t == 50:
            hdg += 45  # simultaneous turn
        lat0 += math.cos(math.radians(hdg)) * spd / 111000 * 10
        lon0 += math.sin(math.radians(hdg)) * spd / (111000 * math.cos(math.radians(lat0))) * 10
        rows.append((f'TRK-{track_id:03d}', ts, lat0, lon0,
                     random.uniform(1000, 35000), spd, hdg % 360, cls))

DB.execute('''
    CREATE TABLE tracks AS
    SELECT * FROM (VALUES %s) t(track_id, ts, lat, lon, alt, speed, heading, true_class)
''' % ','.join(f"('{r[0]}',{r[1]},{r[2]:.4f},{r[3]:.4f},{r[4]:.0f},{r[5]:.1f},{r[6]:.1f},'{r[7]}')" for r in rows))

print(f'Loaded {len(rows)} track records into DuckDB')
print(DB.execute('SELECT true_class, COUNT(DISTINCT track_id) as tracks FROM tracks GROUP BY true_class').df().to_string(index=False))

In [ ]:
# Tool definitions

@tool
def query_tracks_by_time(start_ts: int, end_ts: int) -> str:
    '''Get all tracks active in a time window.
    Args:
        start_ts: Start Unix timestamp
        end_ts: End Unix timestamp
    '''
    df = DB.execute('SELECT DISTINCT track_id FROM tracks WHERE ts BETWEEN ? AND ?', [start_ts, end_ts]).df()
    return json.dumps({'track_ids': df['track_id'].tolist(), 'count': len(df)})

@tool
def compute_kinematics(track_id: str) -> str:
    '''Compute kinematic features for a track: turn rate, acceleration, heading variance.
    Args:
        track_id: Track identifier e.g. TRK-007
    '''
    df = DB.execute('SELECT ts, speed, heading FROM tracks WHERE track_id=? ORDER BY ts', [track_id]).df()
    if len(df) < 2:
        return json.dumps({'error': 'insufficient data'})
    hdg_var = float(df['heading'].std())
    spd_var = float(df['speed'].std())
    turn_rate = float((df['heading'].diff().abs()).mean())
    return json.dumps({'track_id': track_id, 'heading_variance': round(hdg_var, 2),
                       'speed_variance': round(spd_var, 2), 'mean_turn_rate_deg_per_step': round(turn_rate, 2),
                       'is_maneuvering': turn_rate > 5.0})

@tool
def detect_formation_flying(ts: int, radius_km: float = 50.0) -> str:
    '''Find tracks flying in close formation at a given timestamp.
    Args:
        ts: Timestamp to check
        radius_km: Proximity threshold in km
    '''
    df = DB.execute('SELECT track_id, lat, lon FROM tracks WHERE ts=?', [ts]).df()
    formations = []
    for i, row_i in df.iterrows():
        for j, row_j in df.iterrows():
            if i >= j: continue
            dlat = (row_i['lat'] - row_j['lat']) * 111
            dlon = (row_i['lon'] - row_j['lon']) * 111 * math.cos(math.radians(row_i['lat']))
            dist = math.sqrt(dlat**2 + dlon**2)
            if dist < radius_km:
                formations.append({'track_a': row_i['track_id'], 'track_b': row_j['track_id'], 'dist_km': round(dist, 2)})
    return json.dumps({'formations': formations[:10], 'count': len(formations)})

@tool
def check_geofence_violations(lat_min: float, lat_max: float, lon_min: float, lon_max: float) -> str:
    '''Find tracks that entered a restricted lat/lon bounding box.
    Args:
        lat_min, lat_max, lon_min, lon_max: Bounding box coordinates
    '''
    df = DB.execute('''
        SELECT DISTINCT track_id, COUNT(*) as violation_count
        FROM tracks
        WHERE lat BETWEEN ? AND ? AND lon BETWEEN ? AND ?
        GROUP BY track_id
    ''', [lat_min, lat_max, lon_min, lon_max]).df()
    return json.dumps({'violators': df.to_dict('records')})

@tool
def correlate_temporal_patterns(ts: int, window: int = 30) -> str:
    '''Find tracks with simultaneous heading changes at a timestamp.
    Args:
        ts: Center timestamp
        window: Time window in seconds
    '''
    df = DB.execute('''
        SELECT track_id, AVG(heading) as mean_hdg, STDDEV(heading) as hdg_std
        FROM tracks
        WHERE ts BETWEEN ? AND ?
        GROUP BY track_id
        HAVING hdg_std > 10
    ''', [ts - window, ts + window]).df()
    return json.dumps({'coordinated_tracks': df['track_id'].tolist() if len(df) > 1 else [], 'count': len(df)})

TOOLS = [query_tracks_by_time, compute_kinematics, detect_formation_flying,
         check_geofence_violations, correlate_temporal_patterns]
print(f'Tools: {[t.name for t in TOOLS]}')

In [ ]:
# Build agent
try:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE, temperature=0)
    OLLAMA_AVAILABLE = True
    print(f'Connected to Ollama: {OLLAMA_MODEL}')
except Exception as e:
    print(f'Ollama unavailable ({e}) -- demo will use mock responses')
    OLLAMA_AVAILABLE = False

SYSTEM_PROMPT = '''
You are a tactical data analyst. Analyze track data to identify anomalies.

Analysis protocol:
1. Query tracks in the full time window (36000-36990)
2. Check geofence violations in zone (35.3-35.7, -96.8 to -96.2)
3. Detect formations at timestamps 36300, 36600, 36900
4. Compute kinematics for any track with heading variance > 15
5. Correlate temporal patterns at timestamp 36500

Output a ranked table: track_id | anomaly_type | confidence (high/medium/low)
'''

if OLLAMA_AVAILABLE:
    agent = create_react_agent(llm, TOOLS)
    result = agent.invoke({'messages': [HumanMessage(content=SYSTEM_PROMPT + '\nAnalyze all tracks now.')]})
    final_msg = result['messages'][-1].content
    print('\nAgent output (first 500 chars):')
    print(final_msg[:500])
else:
    print('[Mock mode] Agent would analyze 32 tracks across 5 anomaly classes')

In [ ]:
# Ground truth validation
TRUE_ANOMALIES = {
    f'TRK-{i:03d}' for i, (cls, _, _) in
    [(t, *v) for t, v in [(i, list(TRACK_CLASSES.values())[next(j for j, (lo, hi, _) in enumerate(TRACK_CLASSES.values()) if lo <= i < hi)]) for i in range(32)]]
    if cls != 'noise'
}
TRUE_ANOMALIES = {f'TRK-{i:03d}' for i in range(7, 32)}

# Mock agent detections for demo (replace with parsed agent output in production)
DETECTED = {f'TRK-{i:03d}' for i in list(range(7, 26)) + [27, 28, 30]}

tp = len(TRUE_ANOMALIES & DETECTED)
fp = len(DETECTED - TRUE_ANOMALIES)
fn = len(TRUE_ANOMALIES - DETECTED)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print('=== Ground Truth Validation ===')
print(f'True anomalies:  {len(TRUE_ANOMALIES)}')
print(f'Detected:        {len(DETECTED)}')
print(f'True Positives:  {tp}')
print(f'False Positives: {fp}')
print(f'False Negatives: {fn}')
print(f'Precision: {precision:.3f}')
print(f'Recall:    {recall:.3f}')
print(f'F1:        {f1:.3f}')